# Générateur de chunks RAG depuis MySQL — Ask Toby / Mianatra

Ce notebook transforme les lignes de ta base MySQL en **chunks propres, lisibles et contextualisés** pour un système RAG.

Objectif :
- se connecter à une base MySQL locale ;
- lire automatiquement toutes les tables ;
- utiliser les colonnes importantes indiquées par `*` dans ton schéma ;
- remplacer les clés étrangères comme `tenant_id`, `created_by`, `called_by`, etc. par des noms lisibles ;
- transformer chaque ligne en un vrai chunk en français ;
- remplacer les valeurs `NULL` par `non défini` ;
- exporter les chunks en `JSONL` et `CSV`, prêts pour embeddings / vector database.


## 1. Installer les dépendances

Exécute cette cellule une seule fois.  
Si tu es dans VS Code ou Jupyter Notebook, garde la ligne `%pip`.


In [ ]:
%pip install -q pandas sqlalchemy pymysql python-dotenv tqdm pydantic tiktoken

# Optionnel pour embeddings + recherche hybride locale ensuite :
# %pip install -q sentence-transformers numpy rank-bm25 scikit-learn


## 2. Imports et configuration

Tu peux mettre tes identifiants MySQL directement ici, ou créer un fichier `.env` à côté du notebook.

Exemple `.env` :

```env
MYSQL_HOST=localhost
MYSQL_PORT=3306
MYSQL_USER=root
MYSQL_PASSWORD=ton_mot_de_passe
MYSQL_DATABASE=tenant_management
```


In [ ]:
from __future__ import annotations

import os
import json
import math
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect, text
from tqdm.auto import tqdm

load_dotenv()

MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "tenant_management")

OUTPUT_DIR = Path("rag_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Base ciblée :", MYSQL_DATABASE)
print("Dossier de sortie :", OUTPUT_DIR.resolve())


## 3. Connexion à MySQL local

Si ton mot de passe n'est pas dans `.env`, le notebook te le demandera sans l'afficher.


In [ ]:
import getpass

if not MYSQL_PASSWORD:
    MYSQL_PASSWORD = getpass.getpass("Mot de passe MySQL : ")

connection_url = (
    f"mysql+pymysql://{quote_plus(MYSQL_USER)}:{quote_plus(MYSQL_PASSWORD)}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}?charset=utf8mb4"
)

engine = create_engine(connection_url, pool_pre_ping=True)

with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE() AS db_name, VERSION() AS mysql_version"))
    print(dict(result.fetchone()._mapping))

print("Connexion MySQL OK")


## 4. Lire toutes les tables

Le notebook lit toutes les tables existantes dans la base. Les tables vides seront ignorées lors de la génération des chunks.


In [ ]:
inspector = inspect(engine)
table_names = inspector.get_table_names()

print(f"Tables détectées ({len(table_names)}) :")
for t in table_names:
    print("-", t)

tables: Dict[str, pd.DataFrame] = {}

for table in tqdm(table_names, desc="Lecture des tables"):
    try:
        df = pd.read_sql(f"SELECT * FROM `{table}`", engine)
        tables[table] = df
        print(f"{table}: {len(df)} ligne(s), {len(df.columns)} colonne(s)")
    except Exception as e:
        print(f"Impossible de lire {table}: {e}")


## 5. Colonnes importantes

Ces colonnes viennent de ton fichier de schéma où tu avais marqué les attributs importants avec `*`.

Même si une table n'est pas dans cette liste, le notebook peut quand même générer un chunk avec les colonnes utiles.


In [ ]:
IMPORTANT_COLUMNS: Dict[str, List[str]] = {
    "tenants": [
        "code", "name", "status", "contract_start_date", "contract_end_date", "address",
        "email", "phone", "domain_name", "domain_created_date", "domain_status",
        "additional_info", "logo_url", "director1_name", "director1_email", "director1_phone",
        "director2_name", "director2_email", "director2_phone",
    ],
    "data_registrations": [
        "tenant_id", "title", "description", "data_obtained_date",
        "data_entry_completed_date", "notes", "status",
    ],
    "formation_assistance": [
        "tenant_id", "type", "portal", "location", "scheduled_date", "scheduled_time",
        "duration_hours", "completion_date", "completion_time", "actual_duration_hours",
        "description", "feedback", "google_drive_link", "trainer_name",
        "satisfaction_form_uploaded", "notes", "status", "roles",
        "participants_present", "participants_roles", "real_duration_hours",
        "user_feedbacks", "feedback_document_url", "reference_link", "completion_notes",
    ],
    "formation_participants": [
        "formation_id", "participant_name", "participant_role", "is_trainer",
    ],
    "calls": [
        "tenant_id", "caller_name", "interlocutor_name", "called_by", "call_date",
        "call_time", "duration_seconds", "call_type", "subject", "details",
        "follow_up_required", "follow_up_date",
    ],
    "tenant_interactions": [
        "tenant_id", "interaction_type", "title", "interaction_date", "interaction_time",
        "location", "mianatra_team", "tenant_representatives", "situation",
        "discussed_points", "needs_identified", "decisions", "next_actions",
        "follow_up_date", "status", "priority",
    ],
    "data_items": [
        "tenant_id", "title", "description", "is_default", "status", "notes",
    ],
    "progression_tracker": [
        "tenant_id", "stage1_completed", "stage1_domain_name", "stage2_completed",
        "stage2_dns_creation_date", "stage2_dns_passation_date", "stage3_completed",
        "stage4_completed", "stage4_data_entry_date_start", "stage4_data_entry_date_end",
        "stage4_data_types", "stage5_completed", "stage5_formations_finished",
        "stage5_satisfaction_form_received", "stage5_handover_completed", "stage5_handover_date",
    ],
}

TECHNICAL_COLUMNS = {
    "password", "password_reset_token", "password_reset_expires",
    "failed_login_attempts", "blocked_until", "last_login_at",
    "created_at", "updated_at", "deleted_at", "deleted_by", "updated_by",
}

SENSITIVE_COLUMNS = {
    "password", "password_reset_token",
}

print("Tables avec colonnes importantes :", list(IMPORTANT_COLUMNS.keys()))


## 6. Labels français pour rendre les chunks naturels

Au lieu de produire `tenant_id: 6`, on veut produire quelque chose comme :  
`établissement concerné : L'Ile des Enfants (MPIA-0006)`.


In [ ]:
COLUMN_LABELS = {
    "id": "identifiant",
    "code": "code",
    "name": "nom",
    "username": "nom d'utilisateur",
    "email": "email",
    "phone": "téléphone",
    "role": "rôle",
    "status": "statut",
    "tenant_id": "établissement concerné",
    "created_by": "créé par",
    "called_by": "appel traité par",
    "uploaded_by": "importé par",
    "changed_by": "modifié par",
    "formation_id": "formation ou assistance liée",
    "interaction_id": "interaction liée",
    "file_id": "fichier lié",
    "data_item_id": "donnée demandée liée",
    "contract_start_date": "date de début du contrat",
    "contract_end_date": "date de fin du contrat",
    "address": "adresse",
    "domain_name": "nom de domaine",
    "domain_created_date": "date de création du domaine",
    "domain_status": "statut du domaine",
    "additional_info": "information supplémentaire",
    "logo_url": "logo",
    "director1_name": "nom du premier responsable",
    "director1_email": "email du premier responsable",
    "director1_phone": "téléphone du premier responsable",
    "director2_name": "nom du deuxième responsable",
    "director2_email": "email du deuxième responsable",
    "director2_phone": "téléphone du deuxième responsable",
    "type": "type",
    "portal": "portail ou accès",
    "location": "lieu",
    "scheduled_date": "date prévue",
    "scheduled_time": "heure prévue",
    "duration_hours": "durée prévue en heures",
    "completion_date": "date de réalisation",
    "completion_time": "heure de réalisation",
    "actual_duration_hours": "durée réelle en heures",
    "real_duration_hours": "durée réelle en heures",
    "description": "description",
    "feedback": "retour",
    "google_drive_link": "lien Google Drive",
    "trainer_name": "formateur",
    "participants_present": "participants présents",
    "participants_roles": "rôles des participants",
    "notes": "notes",
    "caller_name": "appelant",
    "interlocutor_name": "interlocuteur",
    "call_date": "date d'appel",
    "call_time": "heure d'appel",
    "duration_seconds": "durée de l'appel en secondes",
    "call_type": "type d'appel",
    "subject": "sujet",
    "details": "détails",
    "follow_up_required": "suivi nécessaire",
    "follow_up_date": "date de suivi",
    "interaction_type": "type d'interaction",
    "title": "titre",
    "interaction_date": "date d'interaction",
    "interaction_time": "heure d'interaction",
    "mianatra_team": "équipe Mianatra",
    "tenant_representatives": "représentants de l'établissement",
    "situation": "situation",
    "discussed_points": "points discutés",
    "needs_identified": "besoins identifiés",
    "decisions": "décisions",
    "next_actions": "prochaines actions",
    "priority": "priorité",
    "is_default": "donnée par défaut",
    "stage1_completed": "étape 1 contrat signé",
    "stage1_domain_name": "domaine de l'étape 1",
    "stage2_completed": "étape 2 DNS terminée",
    "stage2_dns_creation_date": "date de création DNS",
    "stage2_dns_passation_date": "date de passation DNS",
    "stage3_completed": "étape 3 collecte des données terminée",
    "stage4_completed": "étape 4 saisie des données terminée",
    "stage4_data_entry_date_start": "date début saisie des données",
    "stage4_data_entry_date_end": "date fin saisie des données",
    "stage4_data_types": "types de données saisies",
    "stage5_completed": "étape 5 formation terminée",
    "stage5_formations_finished": "formations terminées",
    "stage5_satisfaction_form_received": "formulaire de satisfaction reçu",
    "stage5_handover_completed": "handover terminé",
    "stage5_handover_date": "date de handover",
}

TABLE_LABELS = {
    "users": "utilisateur interne",
    "tenants": "établissement / tenant",
    "data_registrations": "enregistrement de données",
    "domain_availability": "disponibilité de domaine",
    "formation_assistance": "formation ou assistance",
    "formation_participants": "participant à une formation",
    "calls": "appel",
    "tenant_interactions": "interaction avec un établissement",
    "tenant_interaction_audit_logs": "historique de modification d'interaction",
    "file_uploads": "fichier importé",
    "tenant_interaction_attachments": "pièce jointe d'interaction",
    "data_items": "donnée demandée",
    "data_item_attachments": "pièce jointe de donnée demandée",
    "progression_tracker": "suivi de progression",
    "tenant_contract_renewals": "renouvellement de contrat",
}

def label(col: str) -> str:
    return COLUMN_LABELS.get(col, col.replace("_", " "))


## 7. Construire les dictionnaires de correspondance pour remplacer les clés étrangères

Ici, on prépare les lookups :
- `tenant_id` devient `Nom établissement (Code)` ;
- `created_by`, `called_by`, etc. deviennent le nom de l'utilisateur ;
- `formation_id`, `interaction_id`, `file_id`, etc. deviennent un libellé humain.


In [ ]:
def is_null(value: Any) -> bool:
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except Exception:
        pass
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def clean_scalar(value: Any) -> Any:
    if is_null(value):
        return None
    if isinstance(value, float) and value.is_integer():
        return int(value)
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return value


def as_int_or_none(value: Any) -> Optional[int]:
    if is_null(value):
        return None
    try:
        return int(value)
    except Exception:
        return None


def get_df(table: str) -> pd.DataFrame:
    return tables.get(table, pd.DataFrame())

# Tenants
TENANT_LOOKUP: Dict[int, Dict[str, Any]] = {}
if not get_df("tenants").empty:
    for _, row in get_df("tenants").iterrows():
        tid = as_int_or_none(row.get("id"))
        if tid is not None:
            TENANT_LOOKUP[tid] = {
                "id": tid,
                "name": clean_scalar(row.get("name")),
                "code": clean_scalar(row.get("code")),
                "status": clean_scalar(row.get("status")),
                "domain_name": clean_scalar(row.get("domain_name")),
            }

# Users
USER_LOOKUP: Dict[int, Dict[str, Any]] = {}
if not get_df("users").empty:
    for _, row in get_df("users").iterrows():
        uid = as_int_or_none(row.get("id"))
        if uid is not None:
            USER_LOOKUP[uid] = {
                "id": uid,
                "name": clean_scalar(row.get("name")) or clean_scalar(row.get("username")),
                "email": clean_scalar(row.get("email")),
                "role": clean_scalar(row.get("role")),
            }

# Formations / Assistances
FORMATION_LOOKUP: Dict[int, str] = {}
if not get_df("formation_assistance").empty:
    for _, row in get_df("formation_assistance").iterrows():
        fid = as_int_or_none(row.get("id"))
        tenant = TENANT_LOOKUP.get(as_int_or_none(row.get("tenant_id")) or -1, {})
        tenant_label = f"{tenant.get('name', 'non défini')} ({tenant.get('code', 'non défini')})"
        date = clean_scalar(row.get("scheduled_date")) or clean_scalar(row.get("completion_date")) or "date non définie"
        typ = clean_scalar(row.get("type")) or "type non défini"
        status = clean_scalar(row.get("status")) or "statut non défini"
        if fid is not None:
            FORMATION_LOOKUP[fid] = f"{typ} pour {tenant_label}, date {date}, statut {status}"

# Interactions
INTERACTION_LOOKUP: Dict[int, str] = {}
if not get_df("tenant_interactions").empty:
    for _, row in get_df("tenant_interactions").iterrows():
        iid = as_int_or_none(row.get("id"))
        tenant = TENANT_LOOKUP.get(as_int_or_none(row.get("tenant_id")) or -1, {})
        tenant_label = f"{tenant.get('name', 'non défini')} ({tenant.get('code', 'non défini')})"
        title = clean_scalar(row.get("title")) or "interaction sans titre"
        date = clean_scalar(row.get("interaction_date")) or "date non définie"
        if iid is not None:
            INTERACTION_LOOKUP[iid] = f"{title} avec {tenant_label}, date {date}"

# Files
FILE_LOOKUP: Dict[int, str] = {}
if not get_df("file_uploads").empty:
    for _, row in get_df("file_uploads").iterrows():
        fid = as_int_or_none(row.get("id"))
        name = clean_scalar(row.get("original_filename")) or clean_scalar(row.get("file_name")) or "fichier sans nom"
        if fid is not None:
            FILE_LOOKUP[fid] = str(name)

# Data items
DATA_ITEM_LOOKUP: Dict[int, str] = {}
if not get_df("data_items").empty:
    for _, row in get_df("data_items").iterrows():
        did = as_int_or_none(row.get("id"))
        tenant = TENANT_LOOKUP.get(as_int_or_none(row.get("tenant_id")) or -1, {})
        tenant_label = f"{tenant.get('name', 'non défini')} ({tenant.get('code', 'non défini')})"
        title = clean_scalar(row.get("title")) or "donnée sans titre"
        if did is not None:
            DATA_ITEM_LOOKUP[did] = f"{title} pour {tenant_label}"

print(f"Tenants résolus : {len(TENANT_LOOKUP)}")
print(f"Utilisateurs résolus : {len(USER_LOOKUP)}")
print(f"Formations/assistances résolues : {len(FORMATION_LOOKUP)}")
print(f"Interactions résolues : {len(INTERACTION_LOOKUP)}")


## 8. Fonctions de formatage des valeurs et de résolution des IDs

Les `NULL` deviennent `non défini`.  
Les booléens deviennent `oui` / `non`.  
Les IDs étrangers deviennent des noms lisibles.


In [ ]:
def format_value(value: Any) -> str:
    """Convertit une valeur SQL/Pandas en texte propre pour le chunk."""
    if is_null(value):
        return "non défini"

    value = clean_scalar(value)

    if isinstance(value, bool):
        return "oui" if value else "non"

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        # Certains booléens MySQL arrivent comme 0/1.
        return str(value)

    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)

    text_value = str(value).strip()
    return text_value if text_value else "non défini"


def format_bool_like(value: Any) -> str:
    if is_null(value):
        return "non défini"
    if value in [1, "1", True, "true", "True", "oui", "yes"]:
        return "oui"
    if value in [0, "0", False, "false", "False", "non", "no"]:
        return "non"
    return format_value(value)


def tenant_label_from_id(value: Any) -> str:
    tid = as_int_or_none(value)
    if tid is None:
        return "non défini"
    tenant = TENANT_LOOKUP.get(tid)
    if not tenant:
        return f"tenant inconnu avec id {tid}"
    return f"{tenant.get('name') or 'non défini'} ({tenant.get('code') or 'non défini'})"


def user_label_from_id(value: Any) -> str:
    uid = as_int_or_none(value)
    if uid is None:
        return "non défini"
    user = USER_LOOKUP.get(uid)
    if not user:
        return f"utilisateur inconnu avec id {uid}"
    name = user.get("name") or "non défini"
    role = user.get("role") or "non défini"
    return f"{name}, rôle {role}"


def resolve_foreign_key(col: str, value: Any) -> str:
    """Remplace une clé étrangère par un texte humain."""
    if col == "tenant_id":
        return tenant_label_from_id(value)
    if col in {"created_by", "updated_by", "deleted_by", "called_by", "uploaded_by", "changed_by"}:
        return user_label_from_id(value)
    if col == "formation_id":
        fid = as_int_or_none(value)
        return FORMATION_LOOKUP.get(fid, f"formation inconnue avec id {fid}") if fid is not None else "non défini"
    if col == "interaction_id":
        iid = as_int_or_none(value)
        return INTERACTION_LOOKUP.get(iid, f"interaction inconnue avec id {iid}") if iid is not None else "non défini"
    if col == "file_id":
        fid = as_int_or_none(value)
        return FILE_LOOKUP.get(fid, f"fichier inconnu avec id {fid}") if fid is not None else "non défini"
    if col == "data_item_id":
        did = as_int_or_none(value)
        return DATA_ITEM_LOOKUP.get(did, f"donnée demandée inconnue avec id {did}") if did is not None else "non défini"
    return format_value(value)


def is_fk_column(col: str) -> bool:
    return col == "tenant_id" or col.endswith("_by") or col in {"formation_id", "interaction_id", "file_id", "data_item_id"}


## 9. Sélectionner les colonnes à mettre dans chaque chunk

Priorité :
1. colonnes marquées `*` dans ton schéma ;
2. sinon, colonnes utiles automatiquement ;
3. on évite les champs techniques et sensibles.


In [ ]:
def useful_columns_for_table(table: str, df: pd.DataFrame) -> List[str]:
    existing_cols = list(df.columns)

    if table in IMPORTANT_COLUMNS:
        cols = [c for c in IMPORTANT_COLUMNS[table] if c in existing_cols and c not in SENSITIVE_COLUMNS]
    else:
        cols = [c for c in existing_cols if c not in TECHNICAL_COLUMNS and c not in SENSITIVE_COLUMNS and c != "id"]

    # Garder le contexte tenant si disponible, même si la table n'est pas dans IMPORTANT_COLUMNS.
    if "tenant_id" in existing_cols and "tenant_id" not in cols:
        cols = ["tenant_id"] + cols

    # Garder l'auteur si utile.
    for author_col in ["created_by", "called_by", "uploaded_by", "changed_by"]:
        if author_col in existing_cols and author_col not in cols:
            cols.append(author_col)

    # Retirer les doublons en gardant l'ordre.
    deduped = []
    for c in cols:
        if c not in deduped:
            deduped.append(c)
    return deduped

for table, df in tables.items():
    if not df.empty:
        print(table, "=>", useful_columns_for_table(table, df))


## 10. Templates de chunks par type de table

Chaque ligne devient un **chunk complet**.  
Le chunk garde le contexte du tenant et contient les informations importantes sous forme de phrase.


In [ ]:
def get_tenant_metadata_from_row(table: str, row: pd.Series) -> Dict[str, Any]:
    """Retourne tenant_id, tenant_name, tenant_code si possible."""
    if table == "tenants":
        return {
            "tenant_id": clean_scalar(row.get("id")),
            "tenant_name": clean_scalar(row.get("name")),
            "tenant_code": clean_scalar(row.get("code")),
        }

    if "tenant_id" in row.index:
        tid = as_int_or_none(row.get("tenant_id"))
        tenant = TENANT_LOOKUP.get(tid or -1, {})
        return {
            "tenant_id": tid,
            "tenant_name": tenant.get("name"),
            "tenant_code": tenant.get("code"),
        }

    # Cas formation_participants : la table pointe vers formation_id, puis la formation pointe vers tenant_id.
    if table == "formation_participants" and "formation_id" in row.index:
        fid = as_int_or_none(row.get("formation_id"))
        fa = get_df("formation_assistance")
        if fid is not None and not fa.empty and "id" in fa.columns:
            matched = fa[fa["id"] == fid]
            if not matched.empty:
                tenant_id = as_int_or_none(matched.iloc[0].get("tenant_id"))
                tenant = TENANT_LOOKUP.get(tenant_id or -1, {})
                return {
                    "tenant_id": tenant_id,
                    "tenant_name": tenant.get("name"),
                    "tenant_code": tenant.get("code"),
                }

    return {"tenant_id": None, "tenant_name": None, "tenant_code": None}


def main_date_from_row(row: pd.Series) -> Optional[str]:
    for col in [
        "call_date", "interaction_date", "scheduled_date", "completion_date",
        "follow_up_date", "contract_start_date", "contract_end_date",
        "stage5_handover_date", "created_at", "updated_at",
    ]:
        if col in row.index and not is_null(row.get(col)):
            return format_value(row.get(col))
    return None


def status_from_row(row: pd.Series) -> Optional[str]:
    if "status" in row.index and not is_null(row.get("status")):
        return format_value(row.get("status"))
    return None


def format_field(col: str, row: pd.Series, bool_columns: Optional[set] = None) -> str:
    bool_columns = bool_columns or set()
    value = row.get(col)
    if col in bool_columns:
        value_text = format_bool_like(value)
    elif is_fk_column(col):
        value_text = resolve_foreign_key(col, value)
    else:
        value_text = format_value(value)
    return f"{label(col)} : {value_text}"


def generic_chunk(table: str, row: pd.Series, cols: List[str]) -> str:
    table_label = TABLE_LABELS.get(table, table)
    row_id = format_value(row.get("id")) if "id" in row.index else "non défini"
    tenant_meta = get_tenant_metadata_from_row(table, row)
    tenant_part = ""
    if tenant_meta.get("tenant_name") or tenant_meta.get("tenant_code"):
        tenant_part = f" pour l'établissement {tenant_meta.get('tenant_name') or 'non défini'} ({tenant_meta.get('tenant_code') or 'non défini'})"
    fields = [format_field(c, row) for c in cols if c in row.index]
    return f"Enregistrement {row_id} de type {table_label}{tenant_part}. " + "; ".join(fields) + "."


def tenant_chunk(row: pd.Series, cols: List[str]) -> str:
    return (
        f"L'établissement {format_value(row.get('name'))} ({format_value(row.get('code'))}) "
        f"a le statut {format_value(row.get('status'))}. "
        f"Son contrat commence le {format_value(row.get('contract_start_date'))} et se termine le {format_value(row.get('contract_end_date'))}. "
        f"Adresse : {format_value(row.get('address'))}. "
        f"Contact : email {format_value(row.get('email'))}, téléphone {format_value(row.get('phone'))}. "
        f"Domaine : {format_value(row.get('domain_name'))}, statut du domaine {format_value(row.get('domain_status'))}, date de création du domaine {format_value(row.get('domain_created_date'))}. "
        f"Informations supplémentaires : {format_value(row.get('additional_info'))}. "
        f"Premier responsable : {format_value(row.get('director1_name'))}, email {format_value(row.get('director1_email'))}, téléphone {format_value(row.get('director1_phone'))}. "
        f"Deuxième responsable : {format_value(row.get('director2_name'))}, email {format_value(row.get('director2_email'))}, téléphone {format_value(row.get('director2_phone'))}."
    )


def formation_chunk(row: pd.Series, cols: List[str]) -> str:
    tenant = tenant_label_from_id(row.get("tenant_id"))
    return (
        f"Pour l'établissement {tenant}, une activité de type {format_value(row.get('type'))} est enregistrée. "
        f"Elle concerne le portail {format_value(row.get('portal'))} et le lieu {format_value(row.get('location'))}. "
        f"La date prévue est {format_value(row.get('scheduled_date'))} à {format_value(row.get('scheduled_time'))}, avec une durée prévue de {format_value(row.get('duration_hours'))} heure(s). "
        f"La réalisation est indiquée le {format_value(row.get('completion_date'))} à {format_value(row.get('completion_time'))}, avec une durée réelle de {format_value(row.get('actual_duration_hours'))} heure(s). "
        f"Description : {format_value(row.get('description'))}. "
        f"Statut : {format_value(row.get('status'))}. "
        f"Formateur(s) : {format_value(row.get('trainer_name'))}. "
        f"Participants présents : {format_value(row.get('participants_present'))}. "
        f"Feedback : {format_value(row.get('feedback'))}. "
        f"Notes : {format_value(row.get('notes'))}. "
        f"Créé par : {resolve_foreign_key('created_by', row.get('created_by'))}."
    )


def call_chunk(row: pd.Series, cols: List[str]) -> str:
    tenant = tenant_label_from_id(row.get("tenant_id"))
    return (
        f"Un appel est enregistré pour l'établissement {tenant}. "
        f"Le {format_value(row.get('call_date'))} à {format_value(row.get('call_time'))}, "
        f"{format_value(row.get('caller_name'))} a échangé avec {format_value(row.get('interlocutor_name'))}. "
        f"Type d'appel : {format_value(row.get('call_type'))}. "
        f"Sujet : {format_value(row.get('subject'))}. "
        f"Détails : {format_value(row.get('details'))}. "
        f"Durée : {format_value(row.get('duration_seconds'))} secondes. "
        f"Suivi nécessaire : {format_bool_like(row.get('follow_up_required'))}. "
        f"Date de suivi : {format_value(row.get('follow_up_date'))}. "
        f"Appel traité par : {resolve_foreign_key('called_by', row.get('called_by'))}."
    )


def interaction_chunk(row: pd.Series, cols: List[str]) -> str:
    tenant = tenant_label_from_id(row.get("tenant_id"))
    return (
        f"Une interaction avec l'établissement {tenant} est enregistrée sous le titre « {format_value(row.get('title'))} ». "
        f"Type : {format_value(row.get('interaction_type'))}. "
        f"Date et heure : {format_value(row.get('interaction_date'))} à {format_value(row.get('interaction_time'))}. "
        f"Lieu : {format_value(row.get('location'))}. "
        f"Équipe Mianatra : {format_value(row.get('mianatra_team'))}. "
        f"Représentants de l'établissement : {format_value(row.get('tenant_representatives'))}. "
        f"Situation : {format_value(row.get('situation'))}. "
        f"Points discutés : {format_value(row.get('discussed_points'))}. "
        f"Besoins identifiés : {format_value(row.get('needs_identified'))}. "
        f"Décisions : {format_value(row.get('decisions'))}. "
        f"Prochaines actions : {format_value(row.get('next_actions'))}. "
        f"Date de suivi : {format_value(row.get('follow_up_date'))}. "
        f"Statut : {format_value(row.get('status'))}. Priorité : {format_value(row.get('priority'))}. "
        f"Créé par : {resolve_foreign_key('created_by', row.get('created_by'))}."
    )


def data_item_chunk(row: pd.Series, cols: List[str]) -> str:
    tenant = tenant_label_from_id(row.get("tenant_id"))
    return (
        f"Pour l'établissement {tenant}, une donnée demandée est enregistrée : « {format_value(row.get('title'))} ». "
        f"Description : {format_value(row.get('description'))}. "
        f"Donnée par défaut : {format_bool_like(row.get('is_default'))}. "
        f"Statut : {format_value(row.get('status'))}. "
        f"Notes : {format_value(row.get('notes'))}. "
        f"Créé par : {resolve_foreign_key('created_by', row.get('created_by'))}."
    )


def progression_chunk(row: pd.Series, cols: List[str]) -> str:
    tenant = tenant_label_from_id(row.get("tenant_id"))
    return (
        f"Le suivi de progression de l'établissement {tenant} indique que l'étape 1 du contrat signé est {format_bool_like(row.get('stage1_completed'))}. "
        f"Le domaine de l'étape 1 est {format_value(row.get('stage1_domain_name'))}. "
        f"L'étape 2 DNS est terminée : {format_bool_like(row.get('stage2_completed'))}. "
        f"Date de création DNS : {format_value(row.get('stage2_dns_creation_date'))}. Date de passation DNS : {format_value(row.get('stage2_dns_passation_date'))}. "
        f"L'étape 3 de collecte des données est terminée : {format_bool_like(row.get('stage3_completed'))}. "
        f"L'étape 4 de saisie des données est terminée : {format_bool_like(row.get('stage4_completed'))}. "
        f"La saisie commence le {format_value(row.get('stage4_data_entry_date_start'))} et finit le {format_value(row.get('stage4_data_entry_date_end'))}. "
        f"Types de données saisies : {format_value(row.get('stage4_data_types'))}. "
        f"L'étape 5 formation est terminée : {format_bool_like(row.get('stage5_completed'))}. "
        f"Formations terminées : {format_bool_like(row.get('stage5_formations_finished'))}. "
        f"Formulaire de satisfaction reçu : {format_bool_like(row.get('stage5_satisfaction_form_received'))}. "
        f"Handover terminé : {format_bool_like(row.get('stage5_handover_completed'))}. "
        f"Date de handover : {format_value(row.get('stage5_handover_date'))}."
    )


def row_to_text_chunk(table: str, row: pd.Series, cols: List[str]) -> str:
    if table == "tenants":
        return tenant_chunk(row, cols)
    if table == "formation_assistance":
        return formation_chunk(row, cols)
    if table == "calls":
        return call_chunk(row, cols)
    if table == "tenant_interactions":
        return interaction_chunk(row, cols)
    if table == "data_items":
        return data_item_chunk(row, cols)
    if table == "progression_tracker":
        return progression_chunk(row, cols)
    return generic_chunk(table, row, cols)


## 11. Générer les chunks avec metadata

Format final d'un chunk :

```json
{
  "chunk_id": "...",
  "text": "phrase complète...",
  "metadata": {
    "source": "mysql:tenant_management.tenants:1",
    "table": "tenants",
    "row_id": 1,
    "tenant_name": "...",
    "tenant_code": "...",
    "status": "active"
  }
}
```


In [ ]:
def stable_chunk_id(table: str, row_id: Any, text_chunk: str) -> str:
    base = f"{MYSQL_DATABASE}:{table}:{row_id}:{text_chunk}"
    return hashlib.sha256(base.encode("utf-8")).hexdigest()[:24]


def row_values_for_metadata(row: pd.Series, cols: List[str]) -> Dict[str, Any]:
    values = {}
    for col in cols:
        if col in row.index:
            if is_fk_column(col):
                values[col] = resolve_foreign_key(col, row.get(col))
            else:
                values[col] = clean_scalar(row.get(col)) if not is_null(row.get(col)) else "non défini"
    return values

chunks: List[Dict[str, Any]] = []

for table, df in tqdm(tables.items(), desc="Génération des chunks"):
    if df.empty:
        continue

    cols = useful_columns_for_table(table, df)

    for _, row in df.iterrows():
        row_id = clean_scalar(row.get("id")) if "id" in row.index else None
        text_chunk = row_to_text_chunk(table, row, cols)
        tenant_meta = get_tenant_metadata_from_row(table, row)
        chunk_id = stable_chunk_id(table, row_id, text_chunk)

        metadata = {
            "source": f"mysql:{MYSQL_DATABASE}.{table}:{row_id}",
            "database": MYSQL_DATABASE,
            "table": table,
            "table_label": TABLE_LABELS.get(table, table),
            "row_id": row_id,
            "tenant_id": tenant_meta.get("tenant_id"),
            "tenant_name": tenant_meta.get("tenant_name") or "non défini",
            "tenant_code": tenant_meta.get("tenant_code") or "non défini",
            "status": status_from_row(row) or "non défini",
            "main_date": main_date_from_row(row) or "non défini",
            "important_columns": cols,
            "values": row_values_for_metadata(row, cols),
        }

        chunks.append({
            "chunk_id": chunk_id,
            "text": text_chunk,
            "metadata": metadata,
        })

print(f"Nombre total de chunks générés : {len(chunks)}")

# Aperçu rapide
for c in chunks[:3]:
    print("\n---")
    print(c["text"])
    print(c["metadata"])


## 12. Contrôle qualité des chunks

Cette cellule vérifie :
- qu'aucun chunk n'est vide ;
- que les `NULL` ont bien été remplacés ;
- que les chunks sont assez courts pour être utilisés en RAG.


In [ ]:
def simple_token_estimate(text_value: str) -> int:
    # Approximation simple : 1 token ≈ 4 caractères en moyenne.
    return max(1, len(text_value) // 4)

quality_rows = []
for c in chunks:
    text_value = c["text"]
    quality_rows.append({
        "chunk_id": c["chunk_id"],
        "table": c["metadata"]["table"],
        "tenant": c["metadata"].get("tenant_name"),
        "chars": len(text_value),
        "estimated_tokens": simple_token_estimate(text_value),
        "has_sql_null_word": "NULL" in text_value,
        "is_empty": len(text_value.strip()) == 0,
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df.describe(include="all"))

print("Chunks vides :", int(quality_df["is_empty"].sum()) if not quality_df.empty else 0)
print("Chunks contenant encore le mot SQL NULL :", int(quality_df["has_sql_null_word"].sum()) if not quality_df.empty else 0)

# Afficher les chunks très longs si besoin
long_chunks = quality_df[quality_df["estimated_tokens"] > 1000] if not quality_df.empty else pd.DataFrame()
print("Chunks estimés > 1000 tokens :", len(long_chunks))
if len(long_chunks) > 0:
    display(long_chunks.head(10))


## 13. Exporter les chunks pour le RAG

`JSONL` est le format le plus pratique pour charger ensuite dans LangChain, LlamaIndex, Qdrant, Chroma, FAISS, etc.


In [ ]:
jsonl_path = OUTPUT_DIR / "rag_chunks.jsonl"
json_path = OUTPUT_DIR / "rag_chunks.json"
csv_path = OUTPUT_DIR / "rag_chunks.csv"

# JSONL : 1 chunk par ligne
with jsonl_path.open("w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# JSON complet
with json_path.open("w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

# CSV lisible
flat_rows = []
for c in chunks:
    meta = c["metadata"]
    flat_rows.append({
        "chunk_id": c["chunk_id"],
        "text": c["text"],
        "source": meta.get("source"),
        "database": meta.get("database"),
        "table": meta.get("table"),
        "row_id": meta.get("row_id"),
        "tenant_id": meta.get("tenant_id"),
        "tenant_name": meta.get("tenant_name"),
        "tenant_code": meta.get("tenant_code"),
        "status": meta.get("status"),
        "main_date": meta.get("main_date"),
    })

chunks_df = pd.DataFrame(flat_rows)
chunks_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Export terminé :")
print("-", jsonl_path.resolve())
print("-", json_path.resolve())
print("-", csv_path.resolve())

display(chunks_df.head(10))


## 14. Exemple de lecture des chunks exportés

Cette cellule montre comment relire le fichier `rag_chunks.jsonl`.


In [ ]:
loaded_chunks = []
with jsonl_path.open("r", encoding="utf-8") as f:
    for line in f:
        loaded_chunks.append(json.loads(line))

print("Chunks chargés depuis JSONL :", len(loaded_chunks))
print(json.dumps(loaded_chunks[0], ensure_ascii=False, indent=2) if loaded_chunks else "Aucun chunk")


## 15. Optionnel : créer des embeddings locaux

Cette partie est optionnelle.  
Elle transforme les chunks en vecteurs avec un modèle multilingue, utile pour du RAG en français.

Active `RUN_EMBEDDINGS = True` seulement si tu veux générer les embeddings maintenant.


In [ ]:
RUN_EMBEDDINGS = False

if RUN_EMBEDDINGS:
    import numpy as np
    from sentence_transformers import SentenceTransformer

    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    model = SentenceTransformer(model_name)

    texts = [c["text"] for c in chunks]
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embeddings_path = OUTPUT_DIR / "rag_embeddings.npy"
    np.save(embeddings_path, embeddings)

    print("Embeddings générés :", embeddings.shape)
    print("Fichier :", embeddings_path.resolve())
else:
    print("Embeddings non générés. Mets RUN_EMBEDDINGS = True pour activer.")


## 16. Optionnel : exemple minimal de recherche après embeddings

À utiliser seulement après avoir activé la génération d'embeddings.


In [ ]:
RUN_LOCAL_SEARCH_DEMO = False

if RUN_LOCAL_SEARCH_DEMO:
    import numpy as np
    from sentence_transformers import SentenceTransformer

    embeddings = np.load(OUTPUT_DIR / "rag_embeddings.npy")
    model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

    def search_chunks(question: str, top_k: int = 5):
        q = model.encode([question], normalize_embeddings=True)[0]
        scores = embeddings @ q
        best_idx = scores.argsort()[-top_k:][::-1]
        return [(float(scores[i]), chunks[i]) for i in best_idx]

    results = search_chunks("Quelle école a une assistance sur le paiement ?", top_k=5)
    for score, chunk in results:
        print("\nScore:", round(score, 4))
        print(chunk["text"])
        print(chunk["metadata"])
else:
    print("Démo de recherche désactivée.")


## 17. Installer les dépendances pour la recherche hybride

Cette partie ajoute une recherche plus réaliste pour un RAG :

- **semantic search** : recherche par sens avec embeddings ;
- **keyword search** : recherche par mots-clés avec BM25 ;
- **fusion** : combinaison des deux résultats avec Reciprocal Rank Fusion ;
- **reranking optionnel** : reclassement final avec un cross-encoder.

Exécute cette cellule une seule fois si les dépendances ne sont pas encore installées.


In [ ]:
%pip install -q sentence-transformers numpy rank-bm25 scikit-learn


## 18. Préparer les embeddings et l'index BM25

Cette cellule charge les chunks exportés, crée ou recharge les embeddings, puis prépare l'index BM25 pour la recherche par mots-clés.

Important : si `rag_embeddings.npy` existe déjà, le notebook le recharge directement. Sinon, il génère les embeddings automatiquement.


In [ ]:
import re
import json
import unicodedata
from pathlib import Path
from typing import Iterable

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# Sécurité : si tu redémarres le kernel et que chunks n'existe plus,
# on suppose que OUTPUT_DIR vaut rag_output si la configuration précédente n'est plus en mémoire.
try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path("rag_output")

# on recharge automatiquement le fichier JSONL déjà exporté.
try:
    chunks
except NameError:
    jsonl_path = OUTPUT_DIR / "rag_chunks.jsonl"
    chunks = []
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            chunks.append(json.loads(line))

texts = [c["text"] for c in chunks]
print("Nombre de chunks disponibles :", len(texts))

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings_path = OUTPUT_DIR / "rag_embeddings.npy"
FORCE_REBUILD_EMBEDDINGS = False

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

if embeddings_path.exists() and not FORCE_REBUILD_EMBEDDINGS:
    embeddings = np.load(embeddings_path)
    print("Embeddings rechargés depuis :", embeddings_path.resolve())
else:
    embeddings = embedder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(embeddings_path, embeddings)
    print("Embeddings générés et sauvegardés dans :", embeddings_path.resolve())

print("Shape embeddings :", embeddings.shape)

FRENCH_STOPWORDS = {
    "le", "la", "les", "un", "une", "des", "du", "de", "d", "l", "et", "ou",
    "a", "au", "aux", "en", "dans", "pour", "par", "avec", "sur", "ce", "cet",
    "cette", "ces", "est", "sont", "etre", "être", "non", "oui", "plus", "moins",
    "tenant", "table", "chunk", "code"
}

def strip_accents(value: str) -> str:
    value = unicodedata.normalize("NFD", str(value))
    return "".join(ch for ch in value if unicodedata.category(ch) != "Mn")


def tokenize_for_bm25(text: str) -> list[str]:
    """Tokenisation simple, rapide et adaptée au français."""
    clean = strip_accents(text.lower())
    tokens = re.findall(r"[a-z0-9]+", clean)
    return [t for t in tokens if len(t) > 1 and t not in FRENCH_STOPWORDS]


tokenized_corpus = [tokenize_for_bm25(t) for t in texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Index BM25 prêt.")
print("Exemple tokens chunk 0 :", tokenized_corpus[0][:30] if tokenized_corpus else [])


## 19. Fonctions : semantic search + keyword search + fusion + reranking

La fonction principale est `hybrid_search(question, ...)`.

Elle fait :

1. recherche sémantique avec embeddings ;
2. recherche mots-clés avec BM25 ;
3. fusion des deux listes avec **RRF** ;
4. reranking optionnel avec un modèle cross-encoder multilingue.

Le reranking est plus précis, mais plus lent. Active-le avec `use_reranker=True`.


In [ ]:
from sentence_transformers import CrossEncoder


def metadata_matches(chunk: dict, metadata_filter: dict | None) -> bool:
    """Filtre exact simple sur les métadonnées.

    Exemple : {"table": "formation_assistance", "tenant_code": "MPIA-0006"}
    """
    if not metadata_filter:
        return True
    meta = chunk.get("metadata", {})
    for key, expected in metadata_filter.items():
        current = meta.get(key)
        if str(current).lower() != str(expected).lower():
            return False
    return True


def get_allowed_indices(metadata_filter: dict | None = None) -> np.ndarray:
    if not metadata_filter:
        return np.arange(len(chunks))
    return np.array(
        [i for i, c in enumerate(chunks) if metadata_matches(c, metadata_filter)],
        dtype=int,
    )


def semantic_rank(question: str, top_k: int = 30, metadata_filter: dict | None = None) -> list[dict]:
    allowed = get_allowed_indices(metadata_filter)
    if len(allowed) == 0:
        return []

    q_emb = embedder.encode([question], normalize_embeddings=True)[0]
    scores = embeddings @ q_emb

    allowed_scores = scores[allowed]
    order = np.argsort(allowed_scores)[::-1][:top_k]
    selected = allowed[order]

    return [
        {
            "idx": int(i),
            "rank": rank + 1,
            "semantic_score": float(scores[i]),
        }
        for rank, i in enumerate(selected)
    ]


def keyword_rank(question: str, top_k: int = 30, metadata_filter: dict | None = None) -> list[dict]:
    query_tokens = tokenize_for_bm25(question)
    if not query_tokens:
        return []

    allowed = get_allowed_indices(metadata_filter)
    if len(allowed) == 0:
        return []

    scores = np.array(bm25.get_scores(query_tokens), dtype=float)
    allowed_scores = scores[allowed]
    order = np.argsort(allowed_scores)[::-1][:top_k]
    selected = allowed[order]

    # On garde seulement les résultats qui ont au moins un signal keyword.
    results = []
    for rank, i in enumerate(selected):
        if scores[i] <= 0:
            continue
        results.append({
            "idx": int(i),
            "rank": rank + 1,
            "keyword_score": float(scores[i]),
        })
    return results


def reciprocal_rank_fusion(
    semantic_results: list[dict],
    keyword_results: list[dict],
    semantic_weight: float = 0.65,
    keyword_weight: float = 0.35,
    rrf_k: int = 60,
) -> list[dict]:
    """Fusionne les résultats semantic + keyword avec Reciprocal Rank Fusion.

    RRF donne un bon score aux chunks qui apparaissent haut dans plusieurs listes.
    """
    fused: dict[int, dict] = {}

    def add_results(results: list[dict], source: str, weight: float):
        for r in results:
            idx = r["idx"]
            if idx not in fused:
                fused[idx] = {
                    "idx": idx,
                    "fusion_score": 0.0,
                    "semantic_score": None,
                    "keyword_score": None,
                    "semantic_rank": None,
                    "keyword_rank": None,
                    "matched_by": [],
                }
            fused[idx]["fusion_score"] += weight / (rrf_k + r["rank"])
            fused[idx]["matched_by"].append(source)

            if source == "semantic":
                fused[idx]["semantic_score"] = r.get("semantic_score")
                fused[idx]["semantic_rank"] = r.get("rank")
            elif source == "keyword":
                fused[idx]["keyword_score"] = r.get("keyword_score")
                fused[idx]["keyword_rank"] = r.get("rank")

    add_results(semantic_results, "semantic", semantic_weight)
    add_results(keyword_results, "keyword", keyword_weight)

    return sorted(fused.values(), key=lambda x: x["fusion_score"], reverse=True)


RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
_reranker = None


def get_reranker():
    """Charge le reranker seulement quand on en a besoin."""
    global _reranker
    if _reranker is None:
        _reranker = CrossEncoder(RERANKER_MODEL_NAME)
    return _reranker


def hybrid_search(
    question: str,
    top_k: int = 5,
    candidate_k: int = 30,
    semantic_weight: float = 0.65,
    keyword_weight: float = 0.35,
    use_reranker: bool = False,
    metadata_filter: dict | None = None,
) -> list[dict]:
    """Recherche hybride prête pour RAG.

    Args:
        question: question utilisateur.
        top_k: nombre final de chunks retournés.
        candidate_k: nombre de candidats récupérés par méthode avant fusion.
        semantic_weight: poids de la recherche sémantique dans la fusion.
        keyword_weight: poids de la recherche par mots-clés dans la fusion.
        use_reranker: True pour activer le cross-encoder final.
        metadata_filter: filtre exact sur metadata, par exemple
            {"table": "formation_assistance"} ou {"tenant_code": "MPIA-0006"}.
    """
    semantic_results = semantic_rank(question, top_k=candidate_k, metadata_filter=metadata_filter)
    keyword_results = keyword_rank(question, top_k=candidate_k, metadata_filter=metadata_filter)

    fused = reciprocal_rank_fusion(
        semantic_results=semantic_results,
        keyword_results=keyword_results,
        semantic_weight=semantic_weight,
        keyword_weight=keyword_weight,
    )

    # On rerank seulement les meilleurs candidats fusionnés pour éviter de ralentir.
    fused = fused[:candidate_k]

    if use_reranker and fused:
        reranker = get_reranker()
        pairs = [(question, chunks[item["idx"]]["text"]) for item in fused]
        rerank_scores = reranker.predict(pairs)

        for item, score in zip(fused, rerank_scores):
            item["rerank_score"] = float(score)

        fused = sorted(fused, key=lambda x: x["rerank_score"], reverse=True)
    else:
        for item in fused:
            item["rerank_score"] = None

    final_results = []
    for position, item in enumerate(fused[:top_k], start=1):
        chunk = chunks[item["idx"]]
        final_results.append({
            "position": position,
            "chunk_id": chunk.get("chunk_id"),
            "table": chunk.get("table"),
            "text": chunk.get("text"),
            "metadata": chunk.get("metadata", {}),
            "scores": {
                "fusion_score": item.get("fusion_score"),
                "semantic_score": item.get("semantic_score"),
                "keyword_score": item.get("keyword_score"),
                "rerank_score": item.get("rerank_score"),
                "semantic_rank": item.get("semantic_rank"),
                "keyword_rank": item.get("keyword_rank"),
                "matched_by": item.get("matched_by"),
            }
        })

    return final_results


def print_search_results(results: list[dict], max_text_chars: int = 900):
    if not results:
        print("Aucun résultat trouvé.")
        return

    for r in results:
        meta = r["metadata"]
        print("\n" + "=" * 100)
        print(f"#{r['position']} | {r['chunk_id']} | table={r['table']}")
        print("Tenant :", meta.get("tenant_name", "non défini"), "| Code :", meta.get("tenant_code", "non défini"))
        print("Scores :", json.dumps(r["scores"], ensure_ascii=False, indent=2))
        print("Texte :")
        print(r["text"][:max_text_chars])


## 20. Exemple : recherche hybride prête pour RAG

Ici, on teste une question utilisateur.

- `use_reranker=False` : rapide, bon pour commencer ;
- `use_reranker=True` : plus précis, mais télécharge un modèle de reranking et peut être plus lent sur CPU.


In [ ]:
question = "Quelles assistances ont été faites pour L'Ile des Enfants ?"

results = hybrid_search(
    question=question,
    top_k=5,
    candidate_k=30,
    semantic_weight=0.65,
    keyword_weight=0.35,
    use_reranker=False,  # Mets True pour activer le reranking cross-encoder.
    metadata_filter=None,
)

print("Question :", question)
print_search_results(results)

# Exporter les derniers résultats pour vérification.
hybrid_results_path = OUTPUT_DIR / "hybrid_search_last_results.json"
with hybrid_results_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\nRésultats sauvegardés dans :", hybrid_results_path.resolve())


## 21. Construire un contexte RAG avec les meilleurs chunks

Cette cellule transforme les résultats de recherche en contexte propre à envoyer à un LLM.

Le but est d'obtenir un bloc comme :

```text
[Source 1] ...
[Source 2] ...
Question: ...
```

Tu peux ensuite envoyer ce contexte à Gemini, OpenAI, Ollama, etc.


In [ ]:
def build_rag_context(results: list[dict], max_chars: int = 6000) -> str:
    blocks = []
    total_chars = 0

    for r in results:
        meta = r["metadata"]
        source_header = (
            f"[Source {r['position']}] "
            f"table={r['table']} ; "
            f"chunk_id={r['chunk_id']} ; "
            f"tenant={meta.get('tenant_name', 'non défini')} ; "
            f"code={meta.get('tenant_code', 'non défini')}"
        )
        block = source_header + "\n" + r["text"]

        if total_chars + len(block) > max_chars:
            break

        blocks.append(block)
        total_chars += len(block)

    return "\n\n".join(blocks)


def build_rag_prompt(question: str, results: list[dict]) -> str:
    context = build_rag_context(results)
    return f"""Tu es un assistant qui répond uniquement avec les informations du contexte.
Si l'information n'est pas présente, réponds : \"Je ne trouve pas cette information dans les données disponibles.\"

Contexte :
{context}

Question : {question}

Réponse en français, claire et courte :"""

rag_prompt = build_rag_prompt(question, results)
print(rag_prompt[:3000])


## 22. Résultat attendu

À la fin, tu obtiens :

- `rag_output/rag_chunks.jsonl` : meilleur format pour RAG ;
- `rag_output/rag_chunks.json` : format complet lisible ;
- `rag_output/rag_chunks.csv` : vérification facile dans Excel ;
- `rag_output/rag_embeddings.npy` : embeddings locaux pour la recherche sémantique ;
- `rag_output/hybrid_search_last_results.json` : derniers résultats de recherche hybride.

Chaque ligne de ta base devient un chunk autonome, avec son contexte tenant et ses métadonnées.

La section finale permet ensuite de faire une recherche hybride :

```text
Question utilisateur
        ↓
Semantic search avec embeddings
        +
Keyword search avec BM25
        ↓
Fusion RRF
        ↓
Reranking optionnel
        ↓
Top chunks propres pour le prompt RAG
```
